# 15 – Begehungsnotizen: Fehlerrangliste und bessere Labels

Bisher kam bei jeder Validierung dasselbe raus: Clustering trennt nicht gegen die Ground Truth, der Klassifikator liegt unter der trivialen Rate, der Kennlinienmonitor (Notebook 14) nur ca. 30 % über Zufall. Vermutung jedes Mal: die Labels sind das Problem. Hier wird das geprüft.

- **Bisheriges Label:** die strukturierten Alt-Neu-Spalten der Begehungstabelle (Heizzeiten, Rücklaufsollwerte, Warmwasserzeiten)
- **Bisher ungenutzt:** die Freitextnotiz des Monteurs, vorhanden bei 57 von 66 Begehungen

Auslöser ist Zähler 68956351:

- im September 2024 wurde das Ventil repariert, der Monitor sieht das als größten Ausschlag
- in den Alt-Neu-Spalten stehen nur verstellte Sollwerte
- der Stellmotortausch steht nur im Freitext

Wenn das kein Einzelfall ist, haben wir bisher gegen einen unvollständigen Maßstab gemessen.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import befundcodes as bc
from src import kennlinien_monitor as km
from src import load_begehungen as lb
from src import ternaer as tn

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

METER_DIR = ROOT / "data" / "processed" / "real_meters"

kodiert = bc.kodiere_begehungen()
print(f"{len(kodiert)} Begehungen")
print(f"{int(kodiert['text_vorhanden'].sum())} mit Freitext")
print(f"{int(kodiert['datum'].notna().sum())} mit Datum")
print(f"{int(kodiert['meter'].notna().sum())} mit Zaehlernummer")

## 1 Warum die strukturierten Spalten nicht reichen

Die Begehung zu Zähler 68956351, einmal in den bisher genutzten Spalten und einmal im Freitext.

In [ ]:
fall = kodiert[kodiert["meter"] == 68956351]
alt_neu = [sp for paar in bc._ALT_NEU for sp in paar if sp in fall.columns]
print("Termin:", fall["datum"].iloc[0].date())
print("\nStrukturierte Spalten:")
print(fall[alt_neu].T.to_string())
print("\nFreitext:")
print(fall["sonstiges"].iloc[0])

- **Alt-Neu-Spalten:** Rücklaufsollwert am Tag von 20 auf 18 °C, Warmwasserzeit von durchgehend auf 06:00 bis 22:00. Kein Wort zum Ventil.
- **Freitext:** Stellmotor getauscht, Schmutzfänger stark gefüllt.

Wer nur auf die Spalten schaut, sieht eine reine Reglereinstellung. Tatsächlich wurde mechanisch gearbeitet, und genau darauf reagiert der Monitor drei Tage später.

Das ist kein Fehler der Monteure: Die Alt-Neu-Spalten sind für Sollwerte gedacht, für mechanische Arbeiten gibt es schlicht kein Feld.

## 2 Wie die Freitexte kodiert werden

- Die Notizen sind im Telegrammstil: keine Satzzeichen, Tippfehler, wechselnde Schreibweisen.
- Kodiert wird regelbasiert in `src/befundcodes.py`. Bei 57 Notizen kann man jede Regel von Hand prüfen.
- Jede Fehlerkategorie hat eine Liste von Treffermustern, eine Notiz kann mehrere Kategorien bekommen.

**Verneinungen:**

- Ein reiner Stichworttreffer würde aus „Schmutzfänger fast leer" einen verschmutzten Filter machen.
- Dafür gibt es Vetomuster, die nur in einem kurzen Fenster hinter dem Treffer suchen.
- Ohne Satzzeichen kann man nicht an Satzgrenzen schneiden. Ein Veto, das auch nach links schaut, würde Befunde zu anderen Bauteilen löschen.

In [ ]:
regeln = pd.DataFrame([
    {"kategorie": name,
     "treffer": ", ".join(r["treffer"][:4]),
     "veto_danach": ", ".join(r["veto_danach"]) or "-",
     "veto_davor": ", ".join(r["veto_davor"]) or "-"}
    for name, r in bc.BEFUNDE.items()])
print(regeln.to_string(index=False))

In [ ]:
#die vier Faelle, an denen sich das Veto entscheidet
proben = ["Schmutzfänger stark voll",
          "Schmutzfänger fast leer",
          "Zirk abgeklemmt",
          "Schwerkraft Zirkulation Gleiderheizkörper",
          "rw Pumpoe läuft nicht über Regler at Fühler passt"]
for text in proben:
    treffer = [n for n in bc.BEFUNDE if bc.kodiere_text(text)[n]]
    print(f"{text:50s} -> {treffer}")

**Kontextlabels** (keine Fehler, sondern Anlagenbeschreibung):

- Fußbodenheizung, Heizkörper, Holzfeuerung, Naturumlauf, Baujahr
- wichtig für die Interpretation: ein Haus mit Naturumlauf hat andere Rücklauftemperaturen, ohne dass etwas kaputt ist
- „Naturzirkulation" zählt deshalb als Bauart, nicht als Zirkulationsfehler

## 3 Die Fehlerrangliste

Welche Fehler kommen am häufigsten vor? Nenner sind die 57 Begehungen mit Freitext.

In [ ]:
rang = bc.rangliste(kodiert)
print(rang.to_string(index=False))

verteilung = (kodiert[kodiert["text_vorhanden"]]["n_befunde"]
              .value_counts().sort_index())
print("\nBefunde je Notiz:")
print(verteilung.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(rang["befund"][::-1], rang["n"][::-1], color="#4878a8")
ax.set_xlabel("Begehungen mit diesem Befund (von 57 mit Freitext)")
ax.set_title("Fehlerrangliste aus den Begehungsnotizen")
for y, (n, anteil) in enumerate(zip(rang["n"][::-1], rang["anteil"][::-1])):
    ax.text(n + 0.1, y, f"{n}  ({anteil:.0%})", va="center", fontsize=9)
ax.set_xlim(0, rang["n"].max() + 2.5)
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

| Fehler | Notizen (von 57) |
|---|---|
| Stellantrieb / Regelventil | 9 |
| Reglerparameter | 9 |
| Schmutzfänger | 6 |
| Hydraulik | 5 |
| Zirkulation, Pumpe, Dämmung | je 4 |

Passt zur Größenordnung aus dem Meeting (vier bis fünf Hauptfehler).

Einschränkungen:

- Das ist die Häufigkeit in den Notizen, nicht im Netz. Die Begehungen wurden nicht zufällig ausgewählt.
- 25 der 57 Notizen haben gar keinen Fehlerbefund (siehe Abschnitt 5).

## 4 Was die Alt-Neu-Spalten übersehen

Abgleich je Begehung: Nennt der Freitext einen Befund? Hat sich in den Alt-Neu-Spalten etwas geändert?

In [ ]:
tafel = bc.diskrepanzen(kodiert)
breit = tafel.pivot(index="befund_im_text", columns="sollwert_geaendert",
                    values="n").fillna(0).astype(int)
breit.index = ["kein Befund im Text", "Befund im Text"]
breit.columns = ["Sollwert unverändert", "Sollwert geändert"]
print(breit.to_string())
print(f"\nSumme: {int(breit.values.sum())} Begehungen")

- **6 von 66:** Fehler im Freitext, aber kein Sollwert geändert. Die galten bisher als Begehung ohne Eingriff.
- **18:** Sollwerte geändert, aber kein Befund im Text
- **16:** beide Quellen melden nichts

Die beiden Quellen überlappen nur teilweise, keine ist vollständig. Das erklärt, warum der Eingriffs-Split in Notebook 14 nichts getrennt hat.

## 5 Die Notizen ohne Fehlerbefund

25 der 57 Notizen bekommen kein Fehlerlabel. Entweder beschreibt die Notiz nur das Gebäude, oder die Kodierung hat etwas übersehen. Die Liste wurde von Hand durchgesehen.

In [ ]:
offen = bc.unkodiert(kodiert)
print(f"{len(offen)} Notizen ohne Fehlerlabel\n")
for _, zeile in offen.iterrows():
    kontexte = [k for k in bc.KONTEXT if zeile.get(k)]
    print(f"{str(zeile['sonstiges'])[:95]:97s} {kontexte}")

- Fast alle sind reine Anlagenbeschreibungen (Heizungsart, Baujahr, Beistellofen). Kontextlabel ja, Fehlerlabel nein.
- Zwei Notizen sind unklar: einmal ein Defekt ohne klare Zuordnung, einmal Messwerte ohne Bewertung. Das sind Fälle für eine Rückfrage bei der ÜZ.
- Drei Notizen fallen auf „Anlage aus" (z. B. Leerstand). Wichtig für die Zeitreihen: das erklärt Stillstand, der sonst als Anomalie zählen würde.

## 6 Gegenprobe: trifft der Monitor die mechanischen Eingriffe?

Notebook 14: Monitor gegen alle 58 Begehungstermine, Hebel ca. 1,3, p um 0,09. Der Split nach Sollwertänderung hat nichts getrennt.

Mit den kodierten Freitexten lässt sich sinnvoller teilen. Der Monitor beobachtet das Verhältnis von Durchfluss zu Leistung:

- getauschter Stellmotor, gereinigter Schmutzfänger: die Hydraulik ändert sich, der Monitor muss reagieren
- verschobene Heizzeit: gleiche Kennlinie zu anderen Stunden, der Monitor muss nicht reagieren

Gruppen:

- **mechanisch:** Schmutzfänger, Ventil/Stellantrieb, Pumpe, Zirkulation, Hydraulik
- **Vergleich:** Reglerparameter, Messtechnik, Dämmung und Notizen ohne Fehlerbefund

In [ ]:
serien = tn.lade_serien(METER_DIR)
ergebnis = km.monitor_alle(serien)
termine = {int(k): list(v)
           for k, v in lb.begehung_dates_by_meter().items()}

pruefung = km.validiere(ergebnis["alarme"], termine)
basis = km.zufallsbasis(ergebnis["alarme"], ergebnis["abdeckung"], termine)
print(f"{len(ergebnis['alarme'])} Alarme, {len(pruefung)} Pruefzeilen")
print("\nGesamtquote ueber alle Termine:")
print(km.trefferquote(pruefung).to_string(index=False))
print("\nZufallsbasis:")
print(basis.to_string(index=False))

In [ ]:
MECHANISCH = ["schmutzfaenger", "ventil_stellantrieb", "pumpe",
              "zirkulation", "hydraulik_volumenstrom"]

labels = kodiert[kodiert["datum"].notna() & kodiert["meter"].notna()].copy()
labels["meter"] = labels["meter"].astype(int)
labels["begehung"] = pd.to_datetime(labels["datum"]).dt.normalize()
labels["mechanisch"] = labels[MECHANISCH].any(axis=1)
labels = (labels[["meter", "begehung", "mechanisch", "n_befunde"]]
          .drop_duplicates(subset=["meter", "begehung"]))

gepruft = pruefung.merge(labels, on=["meter", "begehung"], how="left")
erwartet = dict(zip(basis["quelle"], basis["erwartete_quote"]))
print(f"{len(gepruft)} Pruefzeilen, "
      f"{int(gepruft['mechanisch'].sum())} davon mechanisch")

In [ ]:
zeilen = []
for name, maske in [("mechanischer Befund", gepruft["mechanisch"] == True),
                    ("kein mechanischer Befund",
                     gepruft["mechanisch"] == False)]:
    for _, z in km.trefferquote(gepruft[maske]).iterrows():
        erw = erwartet[z["quelle"]]
        test = stats.binomtest(int(z["treffer"]), int(z["faelle"]), erw,
                               alternative="greater")
        zeilen.append({"gruppe": name, "kanal": z["quelle"],
                       "treffer": int(z["treffer"]),
                       "faelle": int(z["faelle"]), "quote": z["quote"],
                       "zufall": erw, "hebel": z["quote"] / erw,
                       "p": test.pvalue})
split = pd.DataFrame(zeilen)
print(split.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
gruppen = ["mechanischer Befund", "kein mechanischer Befund"]
kanaele = ["parameter", "residuum"]
breite = 0.35
x = np.arange(len(gruppen))
farben = {"parameter": "#4878a8", "residuum": "#c25b56"}
for i, kanal in enumerate(kanaele):
    teil = split[split["kanal"] == kanal].set_index("gruppe").loc[gruppen]
    ax.bar(x + i * breite, teil["quote"], breite, label=kanal,
           color=farben[kanal])
    for xi, (q, tr, fa) in enumerate(zip(teil["quote"], teil["treffer"],
                                         teil["faelle"])):
        ax.text(xi + i * breite, q + 0.012, f"{tr}/{fa}", ha="center",
                fontsize=9)
    ax.hlines(erwartet[kanal], -0.25, len(gruppen) - 0.4, colors=farben[kanal],
              linestyles="dashed", linewidth=1.2)
ax.set_xticks(x + breite / 2)
ax.set_xticklabels(gruppen)
ax.set_ylabel("Anteil Termine mit Alarm im Fenster")
ax.set_title("Trefferquote je Begehungsgruppe, gestrichelt die Zufallsbasis")
ax.legend(title="Kanal")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

**Residuenkanal:**

- mechanische Gruppe: 10 von 19 Treffer (0,526 gegen Zufallsbasis 0,287)
- übrige 39 Termine: 10 Treffer (0,256), also Zufallsniveau

**Parameterkanal:** gleicher Trend, aber viel schwächer (0,474 gegen 0,410).

Das passt: der Residuenkanal misst Abweichungen vom gewohnten Arbeitspunkt, und genau der ändert sich bei einem Ventiltausch oder einer Filterreinigung.

Das ist die erste Validierung gegen die Ground Truth im Unsupervised-Teil, die über den Zufall hinauskommt.

## 7 Hält das einem Test stand?

- Liegt die Quote der mechanischen Gruppe über dem Zufall?
- Unterscheiden sich die beiden Gruppen?

In [ ]:
res = split[split["kanal"] == "residuum"].set_index("gruppe")
a = int(res.loc["mechanischer Befund", "treffer"])
b = int(res.loc["mechanischer Befund", "faelle"]) - a
c = int(res.loc["kein mechanischer Befund", "treffer"])
d = int(res.loc["kein mechanischer Befund", "faelle"]) - c
odds, p_fisher = stats.fisher_exact([[a, b], [c, d]], alternative="greater")
print(f"Vierfeldertafel Residuum: [[{a}, {b}], [{c}, {d}]]")
print(f"Odds Ratio {odds:.2f}, einseitiger Fisher p = {p_fisher:.4f}")

par = split[split["kanal"] == "parameter"].set_index("gruppe")
a2 = int(par.loc["mechanischer Befund", "treffer"])
b2 = int(par.loc["mechanischer Befund", "faelle"]) - a2
c2 = int(par.loc["kein mechanischer Befund", "treffer"])
d2 = int(par.loc["kein mechanischer Befund", "faelle"]) - c2
odds2, p2 = stats.fisher_exact([[a2, b2], [c2, d2]], alternative="greater")
print(f"Vierfeldertafel Parameter: [[{a2}, {b2}], [{c2}, {d2}]]")
print(f"Odds Ratio {odds2:.2f}, einseitiger Fisher p = {p2:.4f}")

- **Residuenkanal:** p = 0,024 gegen die Zufallsbasis, Odds Ratio 3,22 im Gruppenvergleich (p = 0,042)
- **Parameterkanal:** trennt nicht (Odds Ratio 1,29, p = 0,43)

Einschränkungen:

- **Kleine Gruppe:** 19 Termine, 10 Treffer. Ein Fall mehr oder weniger verschiebt die Quote um 5 Prozentpunkte.
- **Split nachträglich gewählt:** erst nach dem Nullergebnis aus Notebook 14. Sauber wäre: Gruppe vorab festlegen und an neuen Begehungen prüfen.
- **Einseitige Tests:** zweiseitig p = 0,038 bzw. p = 0,076. Einseitig ist vertretbar, weil die Richtung vorher feststand.
- **Nicht unabhängig:** 58 Termine auf 51 Stationen.

## 8 Gegenprobe: sind die stärksten Alarme nur Zählertausche?

Einwand: Ein neuer Zähler misst anders, die Kennlinie springt, ohne dass an der Station etwas passiert ist.

Prüfen lässt sich das nur grob. Die Zuordnungstabelle zeigt, an welchen Adressen mehrere Zählernummern geführt werden, aber kein Tauschdatum.

In [ ]:
alarme = ergebnis["alarme"].copy()
alarme["betrag"] = alarme["score"].abs()
gewechselt = set(kodiert.loc[kodiert["meter_gewechselt"] == True, "meter"]
                 .dropna().astype(int))
stationen = set(alarme["meter"].astype(int))
top = alarme.sort_values("betrag", ascending=False).head(30)

print(f"Zaehler mit Tausch laut Zuordnung: {len(gewechselt)}")
print(f"davon mit Alarmen: {len(gewechselt & stationen)} "
      f"von {len(stationen)} Alarmstationen")
print(f"Top-30 Alarme auf Tausch-Zaehlern: "
      f"{int(top['meter'].isin(gewechselt).sum())} von 30")
print("\nStaerkste Alarme:")
print(top.head(10)[["meter", "datum", "quelle", "score", "richtung"]]
      .to_string(index=False))

- Zu wenig Daten: nur 2 der 94 Stationen mit Alarmen haben in der Zuordnung mehrere Zählernummern. Damit lässt sich nichts zeigen oder ausschließen.
- Schwaches Indiz dagegen: die zehn stärksten Alarme verteilen sich auf sieben Stationen und drei Kalenderjahre. Ein Tauschartefakt würde sich eher zeitlich bündeln.
- Bleibt offen, es braucht die Tauschdaten mit Datum von der ÜZ.

## 9 Labels je Zähler

Für die weitere Arbeit werden die Labels auf Zählerebene gebraucht. Hat eine Adresse mehrere Begehungen, werden die Labels zusammengeführt (ODER).

In [ ]:
je_zaehler = bc.labels_je_zaehler(kodiert)
print(f"{len(je_zaehler)} Zaehler mit Labels")
print(f"{int((je_zaehler['n_befunde'] > 0).sum())} davon mit mindestens "
      f"einem Fehlerbefund")
print(f"{int(je_zaehler['baujahr'].notna().sum())} mit Baujahr")
print()
print(je_zaehler.head(10).to_string(index=False))

Neu verfügbar:

- Mehrfachlabel für 62 Zähler statt nur „Maßnahme ja/nein", davon 32 mit mindestens einem Fehlerbefund
- Baujahr für 20 Zähler (als Hinweis auf die Heizungsgeneration)

Für einen Entscheidungsbaum reicht das trotzdem nicht: die größten Kategorien haben 9 Fälle, nach Train/Test-Split bleibt nichts Belastbares. Gleiches Problem wie beim überwachten Klassifikator, nur mit besseren Labels.

## Zusammenfassung

**Belastbar:**

- Die Notizen enthalten Information, die in den Alt-Neu-Spalten fehlt: 6 von 66 Begehungen nennen einen Fehler ohne Sollwertänderung.
- Bei 68956351 ist es umgekehrt: Spalten zeigen nur Sollwerte, der Ventiltausch steht nur im Text.
- Fehlerrangliste über 57 Notizen: Stellantrieb/Regelventil und Reglerparameter je 9, Schmutzfänger 6, Hydraulik 5.
- Die Kodierung ist regelbasiert und von Hand gegen alle 57 Notizen geprüft.

**Hauptbefund:**

- Mit den neuen Labels trifft der Residuenkanal 10 von 19 mechanischen Begehungen (0,526 gegen 0,287, p = 0,024), die übrigen 39 liegen auf Zufallsniveau (0,256). Odds Ratio 3,22, p = 0,042.
- Der schwache Gesamtbefund in Notebook 14 lag also mindestens zum Teil am Maßstab, nicht am Monitor.

**Schwach:**

- 19 Termine sind wenig, Tests einseitig, Split nachträglich gewählt, Termine nicht unabhängig
- Die Rangliste beschreibt die Notizen, nicht das Netz (66 Begehungen bei 101 Zählern, nicht zufällig ausgewählt)

**Blockiert:**

- Entscheidungsbaum: zu wenige Labels (größte Klasse 9 Fälle)
- Zählertausch-Check: kein Tauschdatum vorhanden

**Fürs Team:**

- Kategorien so lassen? Zirkulation und Pumpe trennen? Zählt Dämmung als Anlagenfehler?
- Die zwei unklaren Notizen bei der ÜZ nachfragen?
- Tauschdaten mit Datum beschaffen?